# 01 市场数据与大语言模型输入输出

## 本课学习目标

- A. 量化金融主线：开盘价、最高价、最低价、收盘价和成交量（Open, High, Low, Close and Volume，OHLCV）
零基础解释：OHLCV 是一天行情的基础字段。
- B. 大语言模型主线：大语言模型（Large Language Model，LLM）和词元（Token）
零基础解释：LLM 接收文本输入并生成文本输出，Token 是模型处理文本时使用的小片段，本课只展示消息结构，不调用真实模型。
- C. 两条线如何连接：把市场数据和新闻文本转成可检查的表格信号。
- D. 可运行实验：读取 sample_prices.csv，画收盘价曲线，展示一条新闻如何变成消息列表。
- E. 结果解释：观察表格、图表和结构化输出。
- F. 常见错误：把回测收益当成未来收益、把 Mock 当成真实模型。
- G. 课后练习：修改一个参数并重新运行。
- H. 本课术语表：见本课各小节。

## 本课最终输出

一个离线实验输出，不联网、不调用真实模型、不产生真实订单。

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd()
if not (ROOT / "learning").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
DATA = ROOT / "learning" / "data"


## 什么是提示词

### 提示词（Prompt）

零基础解释：提示词是用户交给大语言模型的任务说明。模型会根据提示词、上下文和系统规则生成回答。

提示词不只是一个简单的问题。它可以包含：

- **任务目标**：希望模型完成什么；
- **背景信息**：帮助模型理解上下文；
- **输入数据**：需要模型分析的内容；
- **输出格式**：期望的答案形式，例如 JSON 或纯文本；
- **限制条件**：字数、风格、禁用语等；
- **示例（Few-shot）**：给出一两个期望的输入输出对，让模型模仿。

**模糊提示词和明确提示词的区别：**

| 类型 | 示例 | 问题 |
|------|------|------|
| 模糊 | "分析这条新闻。" | 模型不知道要分析什么、输出什么格式 |
| 明确 | "请判断以下金融新闻对指定公司的短期情绪影响。只返回 positive、neutral 或 negative，并给出一句不超过30字的理由。" | 任务、约束和格式都清楚 |

**重要提醒：**
- 提示词不能保证模型回答正确；
- 提示词不是程序代码，但可以通过结构化约束让输出更容易被程序处理；
- 不同模型对相同提示词可能给出不同回答；
- 好的提示词能降低错误概率，但不能消除幻觉（Hallucination）。

### 消息结构中的三种角色

大语言模型通常使用消息（Message）列表与用户交互，每条消息有一个角色（role）：

| 角色 | 英文 | 说明 | 示例 |
|------|------|------|------|
| 系统消息 | System Message | 设定模型的行为规则和整体约束 | "你是离线教学助手，只解释数据结构。" |
| 用户消息 | User Message | 用户提出的问题或任务 | "请分析这条合成新闻..." |
| 助手消息 | Assistant Message | 模型生成的回复 | "这里不会调用真实模型，只展示结构。" |

下面用对比案例展示模糊提示词和明确提示词的区别。这些代码不调用真实模型，只展示消息结构。

In [ ]:
# 对比：模糊提示词 vs 明确提示词（离线演示，不调用模型）
# 使用一条静态合成新闻作为示例，不依赖前面单元格的变量
example_title = "合成公司报告季度订单增长"
example_body = "这是合成教学数据：管理层报告需求改善，客户保留率稳定。本文不构成投资建议。"

vague_prompt = [
    {"role": "system", "content": "你是一个助手。"},
    {"role": "user", "content": "分析这条新闻。"},
]
clear_prompt = [
    {"role": "system", "content": "你是金融新闻情绪分析助手。只返回 positive、neutral 或 negative，并给出一句不超过30字的理由。"},
    {"role": "user", "content": f"新闻标题：{example_title}\n新闻正文：{example_body}\n请判断情绪。"},
]
print("=" * 50)
print("模糊提示词的消息结构：")
for msg in vague_prompt:
    print(f"  [{msg['role']}] {msg['content'][:60]}...")
print("=" * 50)
print("明确提示词的消息结构：")
for msg in clear_prompt:
    print(f"  [{msg['role']}] {msg['content'][:80]}...")
print("=" * 50)
print("注意：明确提示词包含了任务目标、输出格式和约束条件。")
print("这只是一个结构演示，并未调用任何真实模型。")

## 可运行实验

下面代码只读取 `learning/data` 下的合成数据。

In [ ]:
from learning.src.market_data import load_price_data
import pandas as pd
import matplotlib.pyplot as plt
prices = load_price_data(DATA / "sample_prices.csv")
news = pd.read_csv(DATA / "sample_news.csv")
display(prices.head())
prices.pivot(index="date", columns="ticker", values="close").plot(figsize=(9,4), title="Synthetic close prices")
plt.show()
sample = news.iloc[0]
messages = [
    {"role": "system", "content": "你是离线教学助手，只解释数据结构。"},
    {"role": "user", "content": f"请分析这条合成新闻: {sample['title']}"},
    {"role": "assistant", "content": "这里不会调用真实模型，只展示 Assistant Message 的结构。"},
]
display(messages)

## 结尾总结

你现在应该理解：量化数据和文本模型输出都必须被结构化、校验并按时间对齐。

本课核心收获：
- 认识了 OHLCV 五个行情基础字段；
- 理解了 LLM 通过 Token 处理文本；
- 学会了提示词的构成要素：任务目标、背景信息、输出格式、限制条件、示例；
- 知道模糊提示词和明确提示词的区别；
- 理解了系统消息、用户消息和助手消息三种角色。

哪些结果不能解释为策略一定赚钱：任何图表和收益数字都只是合成数据上的教学结果。

本课使用了哪些英文专业词：
- Large Language Model（LLM，大语言模型）
- Token（词元）
- Prompt（提示词）
- System Message（系统消息）
- User Message（用户消息）
- Assistant Message（助手消息）
- Structured Output（结构化输出）
- OHLCV（开盘价、最高价、最低价、收盘价和成交量）

### 常见错误

1. **日期未排序**：时间序列必须按日期排序，否则图表和计算都出错。
2. **把成交量当作价格**：volume 是成交量，不是价格，单位不同。
3. **认为 Prompt 一定能保证正确回答**：提示词可以约束格式，但模型仍可能出错或编造内容。
4. **混淆消息角色**：系统消息设定规则，用户消息提出问题，助手消息是回答。

### 课后练习

1. **修改提示词**：把明确提示词中的输出格式改为"返回 JSON 格式，包含 label 和 reason 两个字段"，写出新的消息列表。
2. **识别角色**：在本课的消息列表中找到每条消息的角色，说出它们各自的作用。
3. **排查日期问题**：如果 `prices` 数据没有按日期排序，图表会是什么样子？尝试用 `prices.sort_values("date")` 验证。

下一课与本课有什么关系：下一课会在本课的消息结构基础上，引入做多和做空概念，并讲解如何用 Pydantic 校验结构化输出。